# Prompt 4 - Hybrid Prompting

This notebook contains the implementation of the fourth prompting strategy proposed after the initial evaluation. It evaluates Qwen on the reduced `dataset_30` benchmark using a hybrid prompt that combines strong schema grounding, internal planning, and internal verification.


In [ ]:
!pip -q install -U transformers accelerate sentencepiece gdown


In [ ]:
import os
import gdown

os.makedirs("/content/data", exist_ok=True)

LINK_DATASET_30 = "https://drive.google.com/uc?id=1IOvn9rx5-hFES6PU-3Ow8th5h-cl4z6v"
DATASET_PATH = "/content/data/dataset_30.csv"

gdown.download(LINK_DATASET_30, DATASET_PATH, quiet=False)
print("Dataset saved to:", DATASET_PATH)


In [ ]:
import json
import re
import time
from pathlib import Path
from typing import Any, Dict, Optional

import pandas as pd
import torch
from transformers import pipeline

DATASET_PATH = "/content/data/dataset_30.csv"
OUTPUT_PATH = "/content/results_prompt4_qwen_dataset30.csv"
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_NEW_TOKENS = 220
SLEEP_SECONDS = 0.1

FULL_SCHEMA = 'Node properties:\nMovie {posterEmbedding: LIST, url: STRING, runtime: INTEGER, revenue: INTEGER, budget: INTEGER, plotEmbedding: LIST, imdbRating: FLOAT, released: STRING, countries: LIST, languages: LIST, plot: STRING, imdbVotes: INTEGER, imdbId: STRING, year: INTEGER, poster: STRING, movieId: STRING, tmdbId: STRING, title: STRING}\nGenre {name: STRING}\nUser {userId: STRING, name: STRING}\nActor {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nDirector {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nPerson {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nRelationship properties:\nRATED {rating: FLOAT, timestamp: INTEGER}\nACTED_IN {role: STRING}\nDIRECTED {role: STRING}\nThe relationships:\n(:Movie)-[:IN_GENRE]->(:Genre)\n(:User)-[:RATED]->(:Movie)\n(:Actor)-[:ACTED_IN]->(:Movie)\n(:Actor)-[:DIRECTED]->(:Movie)\n(:Director)-[:DIRECTED]->(:Movie)\n(:Director)-[:ACTED_IN]->(:Movie)\n(:Person)-[:ACTED_IN]->(:Movie)\n(:Person)-[:DIRECTED]->(:Movie)'

SYSTEM_PROMPT = (
    "You are an expert Neo4j and Cypher assistant. "
    "Return only Cypher or a single JSON object when explicitly requested. "
    "Never use SQL syntax such as GROUP BY, HAVING, JOIN, or SELECT. "
    "Use Cypher WITH for aggregation. "
    "Use only schema-valid labels, relationship types, and properties."
)

FEW_SHOTS = '''
Example 1
Question: Which movies did user Alice rate?
Cypher:
MATCH (u:User {name: "Alice"})-[:RATED]->(m:Movie)
RETURN m.title

Example 2
Question: What are the top 3 longest movies by runtime?
Cypher:
MATCH (m:Movie)
RETURN m.title, m.runtime
ORDER BY m.runtime DESC
LIMIT 3

Example 3
Question: Which actors starred in movies with a budget above 200 million?
Cypher:
MATCH (a:Actor)-[:ACTED_IN]->(m:Movie)
WHERE m.budget > 200000000
RETURN a.name, m.title, m.budget

Example 4
Question: What genres does the movie "Toy Story" belong to?
Cypher:
MATCH (m:Movie {title: "Toy Story"})-[:IN_GENRE]->(g:Genre)
RETURN g.name

Example 5
Question: Which 5 movies have been rated by the highest number of users?
Cypher:
MATCH (u:User)-[:RATED]->(m:Movie)
WITH m, COUNT(u) AS numUsers
ORDER BY numUsers DESC
LIMIT 5
RETURN m.title AS MovieTitle, numUsers
'''.strip()

PIPE = None

def load_generation_pipeline():
    global PIPE
    if PIPE is not None:
        return PIPE
    torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    PIPE = pipeline(
        "text-generation",
        model=MODEL_NAME,
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    if PIPE.tokenizer.pad_token_id is None:
        PIPE.tokenizer.pad_token_id = PIPE.tokenizer.eos_token_id
    return PIPE

def call_model(prompt: str) -> str:
    pipe = load_generation_pipeline()
    outputs = pipe(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
        pad_token_id=pipe.tokenizer.pad_token_id,
    )
    generated = outputs[0]["generated_text"]
    text = generated[-1]["content"].strip() if isinstance(generated, list) else str(generated).strip()
    time.sleep(SLEEP_SECONDS)
    return text

def extract_json_block(text: str) -> Optional[Dict[str, Any]]:
    if not isinstance(text, str):
        return None
    cleaned = text.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        parsed = json.loads(cleaned)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        return None

def normalize_cypher(text: Any) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def build_prompt(question: str) -> str:
    return f"""
Task:
Translate the natural language question into a correct Cypher query for the given graph schema.

Instructions:
- Use only the node labels, relationship types, and properties explicitly listed in the schema.
- Do not invent labels, relationships, or properties.
- Use literal values exactly as they appear in the question.
- Do not use SQL keywords such as GROUP BY, HAVING, JOIN, or SELECT.
- Use Cypher WITH when aggregation is needed.
- Internally identify the minimal relevant nodes, relationships, and properties before writing the query.
- Internally verify that the query:
  1. matches the question intent,
  2. uses only schema-valid elements,
  3. includes all requested filters, aggregations, ordering, and limits,
  4. returns the information requested by the question.
- If needed, internally correct the query once before answering.
- Return only the final Cypher query, with no explanation.

Schema:
{FULL_SCHEMA}

Few-shot examples:
{FEW_SHOTS}

Question:
{question}
""".strip()

def run_prompt4(resume: bool = True) -> pd.DataFrame:
    df = pd.read_csv(DATASET_PATH)
    completed_ids = set()
    if resume and Path(OUTPUT_PATH).exists():
        existing = pd.read_csv(OUTPUT_PATH)
        if "id" in existing.columns:
            completed_ids = set(existing["id"].astype(str))

    for _, row in df.iterrows():
        if str(row["id"]) in completed_ids:
            continue

        raw_output = ""
        predicted_cypher = ""
        error_message = ""
        parse_ok = False

        try:
            raw_output = call_model(build_prompt(row["question"]))
            parsed = extract_json_block(raw_output)
            if parsed is not None and str(parsed.get("cypher", "")).strip():
                predicted_cypher = str(parsed.get("cypher", "")).strip()
            else:
                predicted_cypher = raw_output.strip()

            parse_ok = bool(predicted_cypher)
            if not parse_ok:
                error_message = "Model output was empty."
        except Exception as exc:
            error_message = str(exc)

        result = {
            "id": row["id"],
            "difficulty": row.get("difficulty", ""),
            "source_type": row.get("source_type", ""),
            "source_row": row.get("source_row", ""),
            "question": row["question"],
            "gold_cypher": row.get("gold_cypher", ""),
            "prompt_name": "prompt_4_hybrid_qwen",
            "model_name": MODEL_NAME,
            "parse_ok": parse_ok,
            "exact_match": normalize_cypher(predicted_cypher) == normalize_cypher(row["gold_cypher"]),
            "predicted_cypher": predicted_cypher,
            "raw_output": raw_output,
            "error_message": error_message,
        }

        row_df = pd.DataFrame([result])
        header = not Path(OUTPUT_PATH).exists()
        row_df.to_csv(OUTPUT_PATH, mode="a", header=header, index=False)
        print(f"[{row['id']}] parse_ok={parse_ok} exact_match={result['exact_match']}")

    return pd.read_csv(OUTPUT_PATH)


In [ ]:
df_results = run_prompt4(resume=True)
df_results.head()
